# Notebook Overview — Extract Spatial Features

## Purpose

This notebook extracts the **spatial feature set** from preprocessed images using the train or test metadata CSV generated earlier in the pipeline.

Spatial features capture intensity distributions, local variability, and structural texture patterns, providing complementary information to gradient-based features for distinguishing real images from AI-generated images.

---

## Inputs

The notebook expects the following:

* Metadata CSV files from the GitHub repository:

  * `metadata/splits/train_metadata.csv`
  * `metadata/splits/test_metadata.csv`

* Preprocessed image archive from Google Drive:

  * `/content/drive/MyDrive/DIP_Project/releases/preprocessed/All_Sources_preprocessed.zip`

---

## Assumptions

* Each metadata CSV contains the columns:

  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* The metadata was created previously by the combine-and-split notebook

* The preprocessed images are stored in a single ZIP archive containing all images at the archive root (no subdirectories)

* The notebook extracts images into a local runtime directory:

  * `data/preprocessed/images/`

* Image paths are constructed using filenames from metadata only

* All preprocessed images are:

  * grayscale
  * resized to **256 × 256**
  * saved as `.png` files

* This notebook does **not** download, split, or preprocess images

* This notebook processes one subset at a time using metadata-driven image selection

* Feature extraction is performed on a **single subset per run** to preserve split integrity

* All outputs are written to the Colab runtime environment and are **not automatically persisted to Google Drive**

---

## Outputs

The following files are written under `metadata/features/`:

* `metadata/features/train_spatial_features.csv`
* `metadata/features/test_spatial_features.csv`

---

## Expected Sizes

* `train_spatial_features.csv` → **14,400 rows**
* `test_spatial_features.csv` → **3,600 rows**

Each row corresponds to one image and includes metadata plus extracted spatial features.

---

## Notes

* The project contains **6 total sources** and **18,000 images**
* Each source contributes **3,000 images**
* The split is performed earlier with exact per-source counts:

  * **train = 2400 per source**
  * **test = 600 per source**

* This notebook extracts only the **spatial feature group (9 features)**

* Spatial features complement:
  * Gradient features (04A)
  * Frequency-domain features (04C)

* All feature outputs are written under `metadata/features/`

* The preprocessed images are distributed as a single compressed archive for convenience

* This notebook operates entirely from structured metadata and does not determine dataset membership from directory traversal

* This notebook must be run **twice**:
  1. Once with `SUBSET_NAME = TRAIN_SUBSET`
  2. Once with `SUBSET_NAME = TEST_SUBSET`

---

**Next step:**  
After both train and test spatial feature files are generated, proceed to **04C_Extract_Frequency_Features.ipynb**.

---
---

### Step 1 — Environment Setup

* Mount Google Drive to access the preprocessed image archive
* Clone the GitHub repository into the local runtime
* Import project configuration
* Select the target subset (`train` or `test`)
* Verify required metadata and ZIP file inputs

In [ ]:
# ============================================
# Step 1: Startup (Environment + Verification)
# ============================================

import os
import sys
import zipfile
from pathlib import Path

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Mount Google Drive to access release ZIP archive
# -------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# -------------------------------------------------
# Clone GitHub repository into Colab runtime
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_SUBSET,
    TEST_SUBSET,
    TRAIN_METADATA_PATH,
    TEST_METADATA_PATH,
    TRAIN_SPATIAL_FEATURES_PATH,
    TEST_SPATIAL_FEATURES_PATH,
    PREPROCESSED_ZIP_PATHS,
    PROCESSED_DATA_DIR,
    FEATURES_METADATA_DIR,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
TRAIN_METADATA_FILE = Path(TRAIN_METADATA_PATH)
TEST_METADATA_FILE = Path(TEST_METADATA_PATH)

TRAIN_OUTPUT_FILE = Path(TRAIN_SPATIAL_FEATURES_PATH)
TEST_OUTPUT_FILE = Path(TEST_SPATIAL_FEATURES_PATH)

PREPROCESSED_ZIP = Path(PREPROCESSED_ZIP_PATHS["all_sources"])

EXTRACTED_IMAGE_DIR = Path(PROCESSED_DATA_DIR) / "images"
FEATURES_DIR = Path(FEATURES_METADATA_DIR)

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Select subset to process
# -------------------------------------------------
SUBSET_NAME = TRAIN_SUBSET   # Either TRAIN_SUBSET or TEST_SUBSET

if SUBSET_NAME == TRAIN_SUBSET:
    METADATA_FILE = TRAIN_METADATA_FILE
    OUTPUT_FILE = TRAIN_OUTPUT_FILE
elif SUBSET_NAME == TEST_SUBSET:
    METADATA_FILE = TEST_METADATA_FILE
    OUTPUT_FILE = TEST_OUTPUT_FILE
else:
    raise ValueError(f"Invalid SUBSET_NAME: {SUBSET_NAME}")

# -------------------------------------------------
# Verify required inputs
# -------------------------------------------------
print("Verifying required inputs...\n")

required_files = [
    METADATA_FILE,
    PREPROCESSED_ZIP,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file: {file_path}")

print("Required input files are present.")

if VERBOSE:
    print(f"Subset selected:  {SUBSET_NAME}")
    print(f"Metadata file:    {METADATA_FILE}")
    print(f"Preprocessed ZIP: {PREPROCESSED_ZIP}")
    print(f"Extracted images: {EXTRACTED_IMAGE_DIR}")
    print(f"Output file:      {OUTPUT_FILE}")

# -------------------------------------------------
# Extract ZIP only if needed
# -------------------------------------------------
existing_png_count = len(list(EXTRACTED_IMAGE_DIR.glob("*.png")))

if existing_png_count > 0:
    print(f"\nExtracted image directory already contains {existing_png_count} PNG files.")
    print("Skipping extraction.")
else:
    print("\nExtracting All_Sources_preprocessed.zip to local runtime...")
    with zipfile.ZipFile(PREPROCESSED_ZIP, "r") as zip_ref:
        zip_ref.extractall(EXTRACTED_IMAGE_DIR)
    print("Extraction complete.")

# -------------------------------------------------
# Verify extracted images
# -------------------------------------------------
image_files = list(EXTRACTED_IMAGE_DIR.glob("*.png"))

if len(image_files) == 0:
    raise FileNotFoundError(
        f"No PNG files found in extracted image directory: {EXTRACTED_IMAGE_DIR}"
    )

print(f"Found {len(image_files)} extracted PNG files.")
print("\nStartup complete.")



### Step 2 — Extract Preprocessed Images (if needed)

* Check whether images have already been extracted in the local runtime
* If not present, extract:

  * `All_Sources_preprocessed.zip`

  into:

  * `data/preprocessed/images/`

* Verify that extracted PNG images are available

In [ ]:
# ============================================
# Step 2: Load Metadata
# ============================================

import pandas as pd

# -------------------------------------------------
# Load selected subset metadata
# -------------------------------------------------
df = pd.read_csv(METADATA_FILE)

print(f"Loaded metadata for subset: {SUBSET_NAME}")
print(f"Metadata shape: {df.shape}")

# -------------------------------------------------
# Basic column verification
# -------------------------------------------------
required_columns = ["filename", "class_label", "source_dataset", "subset"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Metadata file is missing required columns: {missing_columns}")

print("Required metadata columns verified.")

# -------------------------------------------------
# Verify subset consistency
# -------------------------------------------------
unique_subsets = sorted(df["subset"].dropna().unique().tolist())

if unique_subsets != [SUBSET_NAME]:
    raise ValueError(
        f"Metadata subset mismatch. Expected only '{SUBSET_NAME}', found: {unique_subsets}"
    )

if VERBOSE:
    print(f"Subset column verified: {unique_subsets}")

# -------------------------------------------------
# Build image paths from filenames
# -------------------------------------------------
df["image_path"] = df["filename"].apply(lambda x: EXTRACTED_IMAGE_DIR / x)

# -------------------------------------------------
# Verify image files exist
# -------------------------------------------------
missing_images = df.loc[~df["image_path"].apply(lambda p: p.exists()), "filename"].tolist()

if missing_images:
    raise FileNotFoundError(
        "Some image files referenced by metadata were not found in the extracted image directory.\n"
        f"First missing files: {missing_images[:10]}"
    )

print("All metadata-referenced image files were found.")

# -------------------------------------------------
# Display summary
# -------------------------------------------------
if VERBOSE:
    print("\nClass distribution:")
    print(df["class_label"].value_counts())

if VERBOSE:
    print("\nSource distribution:")
    print(df["source_dataset"].value_counts())

    print("\nSample rows:")
    display(df.head())



### Step 3 — Spatial Feature Helper Functions

* Define spatial-domain feature extraction methods
* Compute intensity-based statistics and local structural descriptors

In [ ]:
# ============================================
# Step 3: Spatial Feature Helper Functions
# ============================================

import numpy as np
import cv2
from scipy.stats import entropy
from skimage.filters.rank import entropy as local_entropy
from skimage.morphology import disk

# -------------------------------------------------
# Helper: entropy from histogram
# -------------------------------------------------
def safe_entropy_from_hist(hist, eps=1e-12):
    hist = hist.astype(np.float64)
    hist = hist / (hist.sum() + eps)
    hist = np.clip(hist, eps, None)
    return float(entropy(hist, base=2))

# -------------------------------------------------
# Global image entropy
# -------------------------------------------------
def image_hist_entropy(img, bins=256):
    hist, _ = np.histogram(img.ravel(), bins=bins, range=(0, 255))
    return safe_entropy_from_hist(hist)

# -------------------------------------------------
# Local entropy map
# -------------------------------------------------
def compute_local_entropy(img, window_size=9):
    """
    Compute local entropy using 32-level quantization and
    a disk-shaped neighborhood, matching the original notebook.
    """
    img_uint8 = img.astype(np.uint8)
    img_q = (img_uint8 // 8).astype(np.uint8)   # 32 quantization levels: 0..31

    local_entropy_map = local_entropy(
        img_q,
        disk(window_size // 2)
    ).astype(np.float32)

    return local_entropy_map

# -------------------------------------------------
# Laplacian response
# -------------------------------------------------
def compute_laplacian(img):
    return cv2.Laplacian(img, cv2.CV_32F)

# -------------------------------------------------
# Patch variance statistics
# -------------------------------------------------
def patch_variance_stats(img, patch_size=16):
    h, w = img.shape
    h_crop = (h // patch_size) * patch_size
    w_crop = (w // patch_size) * patch_size

    img_c = img[:h_crop, :w_crop]

    patches = img_c.reshape(
        h_crop // patch_size, patch_size,
        w_crop // patch_size, patch_size
    ).transpose(0, 2, 1, 3)

    patch_vars = patches.reshape(-1, patch_size, patch_size).var(axis=(1, 2))

    return float(patch_vars.mean()), float(patch_vars.std())

# -------------------------------------------------
# Noise residual
# -------------------------------------------------
def compute_noise_residual(img):
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    residual = img.astype(np.float32) - blurred.astype(np.float32)
    return residual

# -------------------------------------------------
# Extract spatial features
# -------------------------------------------------
def extract_spatial_features(img):
    global_entropy = image_hist_entropy(img)

    local_entropy_map = compute_local_entropy(img)
    local_entropy_mean = float(np.mean(local_entropy_map))
    local_entropy_std = float(np.std(local_entropy_map))

    intensity_mean = float(np.mean(img))
    intensity_std = float(np.std(img))

    lap = compute_laplacian(img)
    laplacian_variance = float(np.var(lap))

    patch_var_mean, patch_var_std = patch_variance_stats(img)

    residual = compute_noise_residual(img)
    noise_residual_energy = float(np.mean(residual ** 2))

    features = {
        "Global Entropy": global_entropy,
        "Local Entropy Mean": local_entropy_mean,
        "Local Entropy Std": local_entropy_std,
        "Intensity Mean": intensity_mean,
        "Intensity Std": intensity_std,
        "Laplacian Variance": laplacian_variance,
        "Patch Variance Mean": patch_var_mean,
        "Patch Variance Std": patch_var_std,
        "Noise Residual Energy": noise_residual_energy,
    }

    return features, local_entropy_map, lap, residual

print("Spatial helper functions defined.")



### Step 4 — Preview and Validate Sample Image

* Load a sample image using metadata
* Verify image format and dimensions
* Compute and display spatial features
* Visualize intermediate spatial representations (e.g., local entropy, Laplacian, residual)

In [ ]:
# ============================================
# Step 4: Preview and Validate Sample Image
# ============================================

import matplotlib.pyplot as plt

# -------------------------------------------------
# Select a sample image from metadata
# -------------------------------------------------
sample_row = df.iloc[0]
sample_path = sample_row["image_path"]

if VERBOSE:
    print("Sample metadata row:")
    for key, value in sample_row.items():
        print(f"{key}: {value}")

    print(f"\nSample image path: {sample_path}")

if not sample_path.exists():
    raise FileNotFoundError(f"Sample image not found: {sample_path}")

# -------------------------------------------------
# Load sample image in grayscale
# -------------------------------------------------
sample_img = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

if sample_img is None:
    raise ValueError(f"Could not load sample image: {sample_path}")

if VERBOSE:
    print(f"Loaded sample image shape: {sample_img.shape}")
    print(f"Loaded sample image dtype: {sample_img.dtype}")

# -------------------------------------------------
# Verify expected image properties
# -------------------------------------------------
if len(sample_img.shape) != 2:
    raise ValueError("Expected grayscale image with 2 dimensions.")

if sample_img.shape != (256, 256):
    raise ValueError(f"Expected image shape (256, 256), found {sample_img.shape}")

print("Sample image format verified.")

# -------------------------------------------------
# Compute spatial features for sample image
# -------------------------------------------------
sample_features, local_entropy_map, lap, residual = extract_spatial_features(sample_img)

if VERBOSE:
    print("\nSample spatial features:")
    for key, value in sample_features.items():
        print(f"{key}: {value:.6f}")

    print(f"\nNumber of features extracted: {len(sample_features)}")

# -------------------------------------------------
# Display sample image and derived spatial views
# -------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("Sample Image")
axes[0].axis("off")

axes[1].imshow(local_entropy_map, cmap="gray")
axes[1].set_title("Local Entropy Map")
axes[1].axis("off")

axes[2].imshow(lap, cmap="gray")
axes[2].set_title("Laplacian Response")
axes[2].axis("off")

axes[3].imshow(residual, cmap="gray")
axes[3].set_title("Noise Residual")
axes[3].axis("off")

plt.tight_layout()
plt.show()



### Step 5 — Extract Spatial Features

For each image in the selected subset:

1. Load the image from the extracted image directory
2. Compute the spatial feature set
3. Store features together with metadata fields

The spatial feature group includes:

* Global Entropy
* Local Entropy Mean
* Local Entropy Std
* Intensity Mean
* Intensity Std
* Laplacian Variance
* Patch Variance Mean
* Patch Variance Std
* Noise Residual Energy

In [ ]:
# ============================================
# Step 5: Extract Spatial Features for Subset
# ============================================

from tqdm.notebook import tqdm

# -------------------------------------------------
# Extract features for all images in selected subset
# -------------------------------------------------
records = []
error_count = 0

print(f"Beginning spatial feature extraction for subset: {SUBSET_NAME}")
print(f"Total images to process: {len(df)}\n")

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {SUBSET_NAME} images"):
    image_path = row["image_path"]

    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Could not load image: {image_path}")

        features, _, _, _ = extract_spatial_features(img)

        record = {
            "filename": row["filename"],
            "class_label": row["class_label"],
            "source_dataset": row["source_dataset"],
            "subset": row["subset"],
        }
        record.update(features)
        records.append(record)

    except Exception as e:
        error_count += 1
        if VERBOSE:
            print(f"Skipping {row['filename']} due to error: {e}")

# -------------------------------------------------
# Convert to dataframe
# -------------------------------------------------
spatial_features_df = pd.DataFrame(records)

# -------------------------------------------------
# Verify output structure
# -------------------------------------------------
expected_columns = [
    "filename",
    "class_label",
    "source_dataset",
    "subset",
    "Global Entropy",
    "Local Entropy Mean",
    "Local Entropy Std",
    "Intensity Mean",
    "Intensity Std",
    "Laplacian Variance",
    "Patch Variance Mean",
    "Patch Variance Std",
    "Noise Residual Energy",
]

missing_columns = [col for col in expected_columns if col not in spatial_features_df.columns]
if missing_columns:
    raise ValueError(f"Missing expected output columns: {missing_columns}")

spatial_features_df = spatial_features_df[expected_columns]

# -------------------------------------------------
# Summary
# -------------------------------------------------
print("\nSpatial feature extraction complete.")
print(f"Output shape: {spatial_features_df.shape}")
print(f"Processed images: {len(spatial_features_df)}")
print(f"Expected images:  {len(df)}")

if error_count > 0:
    print(f"Skipped images:   {error_count}")

if len(spatial_features_df) != len(df):
    print("\nWARNING: Some images were skipped during processing.")

if VERBOSE:
    print("\nSample output rows:")
    display(spatial_features_df.head())



### Step 6 — Save Output

* Save extracted features to a subset-specific CSV file
* Verify file creation and row count

In [ ]:
# ============================================
# Step 6: Save Output
# ============================================

# -------------------------------------------------
# Ensure output directory exists
# -------------------------------------------------
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Save spatial feature dataframe
# -------------------------------------------------
spatial_features_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved spatial features to: {OUTPUT_FILE}")

# -------------------------------------------------
# Verify output file exists
# -------------------------------------------------
if not OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Output file was not created: {OUTPUT_FILE}")

# -------------------------------------------------
# Verify saved row count
# -------------------------------------------------
saved_df = pd.read_csv(OUTPUT_FILE)

if len(saved_df) != len(spatial_features_df):
    raise ValueError(
        f"Saved row count mismatch. Expected {len(spatial_features_df)}, found {len(saved_df)}"
    )

print(f"Verified saved file row count: {len(saved_df)}")



### Step 7 — Final Summary

* Display processing summary
* Provide guidance for generating the remaining subset

In [ ]:
# ============================================
# Step 7: Final Summary and Next Step
# ============================================

from IPython.display import display, HTML

print("Spatial feature extraction completed successfully.\n")

print(f"Subset processed : {SUBSET_NAME}")
print(f"Input rows       : {len(df)}")
print(f"Output rows      : {len(spatial_features_df)}")
print(f"Output columns   : {len(spatial_features_df.columns)}")
print(f"Saved file       : {OUTPUT_FILE}")

print("\nFeature columns:")
for col in spatial_features_df.columns[4:]:
    print(f"  - {col}")

# -------------------------------------------------
# Next step message (based on subset just run)
# -------------------------------------------------
if SUBSET_NAME == TRAIN_SUBSET:
    message = """
    <b>Next Step:</b><br>
    Set <code>SUBSET_NAME = TEST_SUBSET</code> in Cell #1 and rerun this notebook to generate <b>test</b> spatial features.
    """
    border_color = "#ff9800"
    bg_color = "#fff3e0"
else:
    message = f"""
    <b>Current Run Complete:</b><br>
    This run generated <b>{OUTPUT_FILE.name}</b> for the <b>test</b> subset.<br>
    Notebook completed.
    """
    border_color = "#4CAF50"
    bg_color = "#E8F5E9"

display(HTML(f"""
<div style="
    padding: 15px;
    border: 2px solid {border_color};
    background-color: {bg_color};
    border-radius: 8px;
    font-size: 16px;
">
{message}
</div>
"""))

